# S4 · Ejercicio 5b — Entrenar y evaluar el modelo OXXO (GBT)

**Curso de Big Data · Dr. Abel Coronado** · Semana 4

> ⚙️ **Esta variante es para: Container (Podman) / Vagrant — Spark 4.0 · Scala 2.13.**  
> Comprueba tu versión con `import pyspark; pyspark.__version__`. Si NO coincide, usa la carpeta de tu plataforma (`container/` o `portable/`).

🎯 **Objetivo.** Entrenar un GBTClassifier sobre el dataset balanceado, evaluar (Accuracy/F1/AUC/matriz) y guardar el modelo en HDFS.

📦 **Datos.** `/data/processed/ml_dataset_final.geoparquet` (salida del 5a).

✅ **Prerrequisitos.** Ejercicio 5a completado.

📝 **Entregable.** Modelo en `hdfs:///models/gbt_oxxo_model` + métricas (AUC ~0.92).

⬇️ **Descarga (Hugging Face):** https://huggingface.co/datasets/abxda/bdp-lab/resolve/main/cuadernos/semana_4/container/03_ClasificacionSupervisada_GBT.ipynb

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# =============================================================================
# Celda 1: Inicialización de Spark y Sedona
# =============================================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, expr, lower, substring, when, rand, lit, sum as _sum
from sedona.spark import SedonaContext

# Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("» Paso 1: Configurando la sesión de Spark para un procesamiento geoespacial intensivo...")
spark = (
    SparkSession.builder.appName("FeatureEngineeringOXXO_HDFS")
    .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
    .config("spark.locality.wait", "0")
    .config("spark.sql.shuffle.partitions", "200") # Optimización para los joins
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
    .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

sedona = SedonaContext.create(spark)
print("✅ Sesión de Spark con Apache Sedona iniciada correctamente.")

In [ ]:
# =============================================================================
# Celda 1: Preparación de Datos para Machine Learning (Versión Corregida)
# =============================================================================
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
import pyspark.sql.functions as F

# --- LÍNEA DE CORRECCIÓN AÑADIDA AQUÍ ---
# Importamos las clases de tipos de datos desde el módulo correcto
from pyspark.sql.types import IntegerType, DoubleType, LongType, FloatType


print("\n» Paso 2: Preparando el dataset para el entrenamiento del modelo.")

# --- 1. Cargar el dataset final si no está en memoria ---
if 'ml_dataset_sdf' not in locals():
    FINAL_DATASET_PATH = "hdfs:///data/processed/ml_dataset_final.geoparquet"
    print(f"   Leyendo el dataset final desde: {FINAL_DATASET_PATH}")
    ml_dataset_sdf = spark.read.format("geoparquet").load(FINAL_DATASET_PATH)

# --- 2. Seleccionar las Features y el Label ---
# Lista de columnas a excluir del conjunto de features
cols_to_exclude = [
    'id', 'klass', 'codigo_act', 'codigo_act_2c', 'clee',
    'nom_estab', 'cve_ent', 'geometry'
]

# Obtenemos la lista de todas las columnas numéricas automáticamente
# AHORA ESTA LÍNEA FUNCIONARÁ CORRECTAMENTE
all_numeric_cols = [f.name for f in ml_dataset_sdf.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType))]

# Filtramos para obtener la lista final de features
feature_cols = [c for c in all_numeric_cols if c not in cols_to_exclude]

print(f"\n   Se usarán {len(feature_cols)} columnas como features.")
print(f"   La variable objetivo (label) es: 'klass'")

print("\n   Guardando la lista de columnas de features para la inferencia...")
features_df_to_save = spark.createDataFrame(feature_cols, "string")
FEATURES_LIST_PATH = "hdfs:///models/gbt_oxxo_model_features"
features_df_to_save.coalesce(1).write.mode("overwrite").text(FEATURES_LIST_PATH)
print(f"✅ Lista de features guardada en: {FEATURES_LIST_PATH}")

# --- 3. Crear el Vector de Features ---
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

model_data_sdf = assembler.transform(ml_dataset_sdf)
model_data_sdf = model_data_sdf.withColumnRenamed("klass", "label")
final_prepared_data = model_data_sdf.select("label", "features")

print("\n   Dataset preparado. Mostrando una muestra con la columna 'features' vectorizada:")
final_prepared_data.show(5, truncate=50)

print("\n✅ Datos listos para el pipeline de Machine Learning.")

In [ ]:
# =============================================================================
# Celda 2: Entrenamiento del Modelo de Clasificación (GBT)
# =============================================================================
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# --- 1. División en Conjuntos de Entrenamiento y Prueba ---
# Usamos una división 80/20, con una semilla para reproducibilidad
(training_data, test_data) = final_prepared_data.randomSplit([0.8, 0.2], seed=42)

print("Datos divididos en conjuntos de entrenamiento y prueba:")
print(f"   Registros de entrenamiento: {training_data.count():,}")
print(f"   Registros de prueba: {test_data.count():,}")

# --- 2. Definir el Modelo ---
# Usamos GBTClassifier, el equivalente de Gradient-Boosted Trees en Spark.
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5,       # Profundidad máxima de los árboles
    maxIter=20        # Número de árboles a entrenar
)

# --- 3. Entrenar el Modelo ---
print("\nEntrenando el modelo GBTClassifier... (Esto puede tardar varios minutos)")
gbt_model = gbt.fit(training_data)
print("✅ Modelo entrenado exitosamente.")

# --- 4. Realizar Predicciones en el Conjunto de Prueba ---
print("\nRealizando predicciones en el conjunto de prueba...")
predictions = gbt_model.transform(test_data)

print("   Muestra de las predicciones:")
predictions.select("label", "prediction", "probability").show(10)

In [ ]:
# =============================================================================
# Celda 3: Evaluación del Rendimiento del Modelo
# =============================================================================

print("\n» Paso 8: Evaluando el rendimiento del modelo entrenado.")

# --- 1. Métricas de Clasificación Múltiple ---
# Usaremos MulticlassClassificationEvaluator para obtener Accuracy, F1, etc.

# Accuracy
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)
print(f"\n   Accuracy: {accuracy:.4f}")

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print(f"   F1 Score: {f1_score:.4f}")

# Precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)
print(f"   Weighted Precision: {precision:.4f}")

# Recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)
print(f"   Weighted Recall: {recall:.4f}")

# --- 2. Métrica de Clasificación Binaria (AUC) ---
# Usaremos BinaryClassificationEvaluator para el Área Bajo la Curva ROC.
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)
print(f"\n   Area Under ROC (AUC): {auc:.4f}")


# --- 3. Matriz de Confusión ---
# La creamos manualmente agrupando los resultados
print("\n   Matriz de Confusión:")
predictions.groupBy('label', 'prediction').count().show()

print("\n✅ Evaluación del modelo completada.")


In [ ]:
# =============================================================================
# Celda 4: Guardado del Modelo Entrenado
# =============================================================================

print("\n» Guardando el modelo GBT entrenado en HDFS.")

# --- 1. Definir la ruta de guardado en HDFS ---
MODEL_PATH_HDFS = "hdfs:///models/gbt_oxxo_model"

# --- 2. Guardar el modelo ---
# Usamos .overwrite() para poder re-entrenar y guardar nuevas versiones del modelo
# sin tener que borrar manualmente la carpeta.
gbt_model.write().overwrite().save(MODEL_PATH_HDFS)

print(f"✅ Modelo guardado exitosamente en: {MODEL_PATH_HDFS}")

In [ ]:
spark.stop()
print("\n✅ Sesión de Spark detenida.")